In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### What do all these import statements mean?

`import torch`
* PyTorch's main library 
* Gives access to tensors (`torch.Tensor`) and core functions such as moving data to GPU/CPU/MPS
* One of the foundation of deep learning

`import torch.nn as nn`
* `torch.nn` is PyTorch's nerual networks building blocks
* It helps us define layers and models
* Lego set of layers you can combine into your own neural networks

`import torch.nn.functional as F`
* Functional interface for activation functions and operations 
* The 'raw ops' toolbox for neural networks

`import torch.optim as optim`
* Optimizers for training models
* Handles updating model weights during training
* The rules for learning, and how the model improves

`from torchvision import datasets, transforms`
* Companion library for vision tasks 
* `datasets` are prebuilt datasets (CIFAR-10, Food-101, MNIST, etc) 
* `transforms` helps with image processing (resize, normalize, augment)
* Saves from manually downloading/preprocessing image datasets
* Convenience kit for CV datasets and image transformations

`froom torch.utils.data import DataLoader` 
* A utility that wraps a dataset and gives batches of data 
* Neural nets train in batches so DataLoader shuffles data, handles batches, and loads efficiently
* The batching machine for datasets

## 0. Goals and Success Criteria

The primary goal of this notebook is to train a Convolutional Neural Network on the Food-101 datasets to be able to classify different food. A goal is to make this classifier generalize well, and also be easy to iterate one

Acceptance criteria
1. Baseline CNN model which we'll create with PyTorch 

    * Top-1 Accuracy >= 40-55%, meaning that at least 40-55% of the time, the model's first guess should be correct 

    * Top-5 Accuracy >= 80%, meaning that for 80% of the images, the correct class should be among the top 5 guesses

    * Achieve this with a small model trained for just a few epochs. This will be the floor we should expect from scratch training

2. Transfer baseline (pretrained ResNet-18, fine-tuned on Food-101)
    * Top-1 Accuracy >= 70%, as with moderate training we should expect to reach this accurac using transfer learning

    * This is the stronger baseline before trying more complex stuff

In other words 
* If your CNN baseline doesn't get near 40% top-1, then something is wrong
 
* If ResNet-18 can't break 70%, maybe our pipeline or training setup has bugs

## 1. Problem Framing
* Task is for multiclass classification 

* Loss: Cross Entropy

* Metrics: Top-1 accuracy (primary) and Top-5 accuracy (secondary), Macro-F1 per class 

* Why: Food 101 has many visually similar classes; top-5 shows if the model's in the right neighborhood when top-1 is wrong


## 2 Data & Splits 

* The dataset we'll be using is the Food101 dataset via `torchvision.datasets`, which provides 75k training points and 25k testing points with a fixed official split 

* We will have to perform our own validation split as the data is split as train/test and **not** train/val/test. Remember, validation sets are used for hyperparameter tuning and early stopping

* Data directory: `./data/food101/` keeps raw data separate from processed/augmented caches

* We would also want to handle occasional corrupted images gracefully (try/except in `__getitem__` and skip).

The following code below will be used as part of `data.py`, but we'll keep it here for now

Notes 

ImageNet
* ImageNet is a massive datasets of around 14M+ labeled images, 1000 categories, and is used to train many pretrained models like ResNet, VGG, etc. 
* When you load a model like `torchvision.models.resnet18(pretrained=True)`, the weights expect inputs which were normalized in the same way they were trained on (ImageNet in this case)
* This is why we need the specific mean and std values, as they come form the ImageNet dataset
* These means and std valures correspond to the average pixel density per color channel (RGB) and how much it varies across the entire dataset

Training Transforms 
* `transforms.Compose` can be thought of as a method for building the pipeline of steps 

    * Each image passes through this pipeline before being given to the model

    * The transformations are applied in the order in which they are list

`RandomResizedCrop(img_size, scale=(0.8, 1.0))`
* Randomly crops the image to a portion (between 80%-100% of the original area), then resizes it to img_size 
* This adds variation in scale and viewpoint

`RandomHoriztonalFlip(p=0.5)` 
* With 50% probability, flips the image left-right
* Useful because flipping doesn't change the class

`ColorJitter(...)`
* Randomly tweaks brightness, contrast, sautration, hue
* Prevents the model from overfitting due to lightning/color conditions

`ToTensor()`
* Converts the image from a PIL image (values 0 - 255 ints) to a PyTorch tensor (values scaled to [0,1])

`Normalize(imagenet_mean, imagenet_std)`
* Standardizes pixel values channel-wise using ImageNet's mean and SD 
* Ensures inputs are on the same scale as what the pretrained models expect

Eval Transforms
* We don't want randomness anymore
* During training we want to add variation in order to encourage generalization 
* Duringe evaluatin we want to keep our inputs as consistent and fair in order to asses our accuracy on the same basis every time

ImageNet eval images are resized to 256x256 before cropping to 224x224 in order to ensure the object is large enough in the image, and it also gives extra border so the crop doesn cut off important parts

`Resize(int(img_size * 1.15))`
* Makes the **shorter** side of the image ~15% larger

`CenterCrop(img_size)`
* Takes the center square patch of size 224x224 (or whatever img_size is)
* Helps with consistency and crops out potentially irrelelvant edges/backgrounds while keeping the main subject 

The Crop methods default to square cropping if we only specify an interger, if a tuple is specified then you get rectangle 


`DataLoader`
- A specific class in PyTorch 
- Is a utility for feeding data into the model during training evaluation
- Can think of as a smart iteratory which retrieves samples from dataset, batches them together, shuffles if needed, and loads them in parallel with multiple worker processes, and hands them to training loop
- Wraps around a `Dataset` and handles batching, shufflinf, and parallel loading

The data lives in `torch.utils.data.Dataset` and we can use ready-made datasets or make our own `Dataset` class. In this case we would be responsible for giving an index, `i` and its corresponding label

`Datasets` like these are composed of the image, and the label (integers)
- But many of these `Dataset` objects also contain an attribute `.classes` which contains a list of class names

In [4]:
import os 
import random 
import numpy as np 
from typing import Tuple, Dict, List

import torch 
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms


In [5]:
# In order to keep our results reproducible, we'll set the seed as below: 

#---------------Reproducibility-----------------
def set_seed(seed: int = 42):
    random.seed(seed) # Python random number generator
    np.random.seed(seed) # for numpy arrays
    torch.manual_seed(seed) #for CPU tensors
    torch.cuda.manual_seed_all(seed) #for GPU tensors
    # Note: making CUDA/MPS fully deterministic can slow things down 
    #torch.backends.cudnn.deterministic = True 
    #torch.backends.cudnn.benchmark = False 

#---------------Transformations-----------------
def build_transforms(img_size: int = 224):
    """
    Train transforms: mild augmentation + normalization 
    Eval transforms: resize + center crop + normalization
    Using ImageNet mean/std so we can swap to pretrained models later if needed
    """
    
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]
    
    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale = (0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02), 
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std)])
    
    eval_transforms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std)
    ])
    
    return train_transforms, eval_transforms

#---------------Splitting helpers-----------------
def split_train_val(
    dataset: torch.utils.data.Dataset,
    val_ratio: float = 0.1,
    seed: int = 42) -> Tuple[Subset, Subset]:
    
    """
    Performs a simple random split of the training portion into a train and validation set
    """
    set_seed(seed)
    n_total = len(dataset)
    n_val = int(n_total * val_ratio)
    n_train = n_total - n_val
    train_subset, val_subset = random_split(dataset, [n_train, n_val])
    return train_subset, val_subset

#---------------Main builder to call from train.py-----------------

def build_datasets(
    data_dir: str = './data',
    img_size: int = 224, 
    batch_size: int = 32, 
    num_workers: int = 2, 
    val_ratio: float = 0.1, 
    seed: int = 42
) -> Tuple[DataLoader, DataLoader, DataLoader, Dict[int, str]]:
    """
    Returns the training, validation, and test dataloaders and an index -> class mapping
    """
    set_seed(seed)
    train_transformations, eval_transformations = build_transforms(img_size)
    
    #Food-101 has fixed "train" and "test" splits, provided by torchvision
    full_train_dataset = datasets.Food101(root=data_dir, split='train', download=True, transform=train_transformations)
    test_dataset = datasets.Food101(root=data_dir, split='test', download=True, transform=eval_transformations)
    
    #Making a  alidation set from the training portion 
    train_subset, val_subset = split_train_val(full_train_dataset, val_ratio=val_ratio, seed=seed)
    
    #It is important that the validation set uses the eval transformations, and not the train transformations
    #One simple way to do this is to make a new dataset with the validation set indices, and use the eval transformations
    #We'll reuse the Food101 metadata (classes, targets) but just use different trasnform. 
    val_base = datasets.Food101(root=data_dir, split='train', download=True, transform=eval_transformations)
    val_dataset = Subset(val_base, val_subset.indices)
    
    #DataLoaders
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    #Class mapping for readability in logs/plots
    #Food 101 exposes classes as a list; the targets are integer indices into this list. 
    idx_to_class = {i: cls for i, cls in enumerate(full_train_dataset.classes)}
    
    return train_loader, val_loader, test_loader, idx_to_class

In [10]:
# ---------- Quick sanity check (run this file directly) ----------
tl, vl, te, idx2cls = build_datasets(
    data_dir='./data',
    img_size=224, 
    batch_size=16, 
    num_workers=2, 
    val_ratio=0.1, 
    seed=42
)

print(f"Train size: {len(tl.dataset)}")
print(f"Validation size: {len(vl.dataset)}")
print(f"Test size: {len(te.dataset)}")

xb, yb = next(iter(tl))
print('One batch shapes:', xb.shape, yb.shape)
print('Example classes:', [idx2cls[yb[i]] for i in range(min(5, len(yb)))])

  3%|▎         | 160M/5.00G [00:20<10:27, 7.71MB/s]  


KeyboardInterrupt: 